# CPU efficiency benchmark

Whisper base/small/medium plus **mHuBERT (FINAL)** over the same few samples, **on CPU**.
Speed ONLY (RTF, wall time) and RAM. NO WER or CER. The goal is to show that the model can run on CPU.

Important: yours is **CTC = a single forward pass**; Whisper is **autoregressive** (token by
token). On CPU that gap widens a lot, and that is the real story. It runs on any runtime, on
the CPU, with no GPU needed.


In [ ]:
!pip install -q peft psutil
!pip uninstall -y -q torchao


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, torch
torch.set_num_threads(os.cpu_count() or 2)
FINAL_DIR = "/content/drive/MyDrive/CLEAR/Phase 1/runs/FINAL"
WHISPER   = ["openai/whisper-base", "openai/whisper-small", "openai/whisper-medium"]
N = 8    # a few samples, you can raise it (medium is slow on CPU, so keep it small)
print("CPU cores:", os.cpu_count(), "| FINAL exists:", os.path.exists(FINAL_DIR + "/head.pt"))


In [ ]:
from datasets import load_dataset
import numpy as np
ds = load_dataset("openslr/librispeech_asr", "clean", split="validation", streaming=True)
CLIPS = []
for i, row in enumerate(ds):
    if i >= N: break
    a = row["audio"]; w = np.asarray(a["array"], np.float32); sr = a["sampling_rate"]
    if sr != 16000:
        w = np.interp(np.linspace(0, len(w)-1, int(len(w)*16000/sr)),
                      np.arange(len(w)), w).astype(np.float32)
    CLIPS.append((w, row["text"]))
TOTAL_SEC = sum(len(w) for w, _ in CLIPS) / 16000
print(f"{len(CLIPS)} clips . total audio {TOTAL_SEC:.0f}s")


In [ ]:
import time, psutil, gc, numpy as np
PROC = psutil.Process()
def rss_mb(): return PROC.memory_info().rss / 1e6
RESULTS = []
def record(name, params_m, secs, ram):
    rtf = secs / TOTAL_SEC
    RESULTS.append({"model": name, "params(M)": round(params_m, 1),
                    f"sec/{N}clips": round(secs, 1), "RTF": round(rtf, 2),
                    "RAM(MB)": round(ram, 0)})
    print(f"  {name:18s} {params_m:6.1f}M | {secs:6.1f}s | RTF {rtf:5.2f} | +{ram:5.0f}MB")


In [ ]:
import torch, time, gc
from transformers import WhisperProcessor, WhisperForConditionalGeneration

for name in WHISPER:
    gc.collect(); r0 = rss_mb()
    proc = WhisperProcessor.from_pretrained(name)
    model = WhisperForConditionalGeneration.from_pretrained(name).to("cpu").eval()
    params = sum(p.numel() for p in model.parameters()) / 1e6
    with torch.no_grad():   # warmup
        f = proc(CLIPS[0][0], sampling_rate=16000, return_tensors="pt").input_features
        model.generate(f, language="en", task="transcribe", max_new_tokens=220)
    peak = rss_mb(); t0 = time.perf_counter()
    with torch.no_grad():
        for w, _ in CLIPS:
            f = proc(w, sampling_rate=16000, return_tensors="pt").input_features
            model.generate(f, language="en", task="transcribe", max_new_tokens=220)
            peak = max(peak, rss_mb())
    dt = time.perf_counter() - t0
    record(name.split("/")[-1], params, dt, peak - r0)
    del model, proc; gc.collect()


In [ ]:
import torch, torch.nn as nn, json, time, gc
from itertools import groupby
from peft import LoraConfig, inject_adapter_in_model
from transformers import HubertModel

CHARS = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ'")
VOCAB = {c: i for i, c in enumerate(CHARS)}
VOCAB["|"], VOCAB["[UNK]"], VOCAB["[PAD]"] = 27, 28, 29
BLANK, UNK = 29, 28; I2C = {i: c for c, i in VOCAB.items()}
def greedy(lg):
    ids = lg.argmax(-1)
    return "".join(I2C[k] for k, _ in groupby(ids.tolist())
                   if k not in (BLANK, UNK)).replace("|", " ").strip()

raw = json.load(open(FINAL_DIR + "/config.json"))
WS = tuple(sorted(int(x) for x in raw.get("ws", (9, 10, 11, 12))))
LL = raw.get("lora_layers") or list(range(1, max(WS)+1)); HID = int(raw.get("hid", 768))
gc.collect(); r0 = rss_mb()
bb = HubertModel.from_pretrained("utter-project/mHuBERT-147",
                                 attn_implementation="sdpa").to("cpu")
lc = LoraConfig(r=int(raw.get("lora_r", 16)), lora_alpha=int(raw.get("lora_alpha", 32)),
                target_modules=["q_proj", "v_proj"], bias="none",
                layers_to_transform=[i-1 for i in LL])
bb = inject_adapter_in_model(lc, bb)
bb.load_state_dict(torch.load(FINAL_DIR + "/adapter.pt", map_location="cpu"), strict=False)
bb.eval()

class Head(nn.Module):
    def __init__(s, n, d, V):
        super().__init__(); s.layer_w = nn.Parameter(torch.zeros(n))
        s.net = nn.Sequential(nn.Linear(d, d), nn.ELU(), nn.Dropout(0.0), nn.Linear(d, V))
    def forward(s, x):
        w = s.layer_w.softmax(0); return s.net((x * w[None, None, :, None]).sum(2))

head = Head(len(WS), HID, len(VOCAB))
sd = torch.load(FINAL_DIR + "/head.pt", map_location="cpu")
if "net.2.weight" in sd and sd["net.2.weight"].shape[0] != sd["net.0.weight"].shape[0]:
    sd["net.3.weight"] = sd.pop("net.2.weight"); sd["net.3.bias"] = sd.pop("net.2.bias")
head.load_state_dict(sd, strict=False); head.eval()
flen = bb._get_feat_extract_output_lengths
params = (sum(p.numel() for p in bb.parameters())
          + sum(p.numel() for p in head.parameters())) / 1e6

def run_mhubert(model_bb, tag, base_rss):
    with torch.no_grad():   # warmup
        x = torch.from_numpy(CLIPS[0][0].astype(np.float32))[None]
        model_bb(x, output_hidden_states=True)
    peak = rss_mb(); t0 = time.perf_counter()
    with torch.no_grad():
        for w, _ in CLIPS:
            x = torch.from_numpy(w.astype(np.float32))[None]
            o = model_bb(x, output_hidden_states=True)
            hs = torch.stack([o.hidden_states[L] for L in WS], 2)
            lg = head(hs.float())[0]
            xl = int(flen(torch.tensor([x.shape[1]]))[0]); _ = greedy(lg[:xl])
            peak = max(peak, rss_mb())
    record(tag, params, time.perf_counter() - t0, peak - base_rss)

run_mhubert(bb, "mHuBERT (fp32)", r0)
try:
    bbq = torch.quantization.quantize_dynamic(bb, {nn.Linear}, dtype=torch.qint8)
    run_mhubert(bbq, "mHuBERT (int8)", r0)
except Exception as e:
    print("int8 skipped:", type(e).__name__, e)


In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
print(df.to_string(index=False))
out = "/content/drive/MyDrive/CLEAR/Phase 1/runs/cpu_bench.csv"
df.to_csv(out, index=False)
print("\nCSV -> " + out)
print("\nNote: RTF<1 means faster than real time. Because CTC is a single forward pass, mHuBERT's "
      "CPU advantage over autoregressive Whisper should be larger than its GPU advantage.")
